# PyPSA tutorial: Refugee Camp

## What is PyPSA?

**PyPSA (Python for Power System Analysis)** is an open-source Python library designed for modeling and optimizing modern power systems. It provides a flexible framework for simulating electricity networks, conducting power flow analysis, performing energy system optimization, market dispatch and multi-sector coupling.

## Key Features

- **Economic Dispatch**: Short-term market dispatch with unit commitment, renewables, storage and multi-carrier conversion.
- **Static Power Flow Analysis**: Build power systems using buses, lines, transformers, generators, loads, and storage etc and perform AC and DC power flow calculations.
- **Time series optimization**: Optimize dispatch and capacity expansion over multiple time periods 
- **Optimal Power Flow (LOPF and SCLOPF)**: Optimize power flow in AC-DC networks using linearized KVL/KCL constraints with optional loss approximations, and accounting for line outage contingencies under N-1 conditions.
- **Flexibile and Multi-Sector**: Model everything from single buildings to continental-scale energy systems, including sector coupling e.g. (electricity, heat, hydrogen etc.)
- **Open source**: Free to use, modify, and extend

## Core Concepts

### 1. Network Object
The `pypsa.Network()` is the central container that holds all components of your power system and optimization results.

### 2. Snapshots
The time index for your simulation - defines when and how often the system is evaluated (e.g., hourly for a year).

### 3. Components
- **Buses**: Nodes where components connect (like electrical junction points)
- **Generators**: Power production (solar, wind, gas, nuclear, etc.)
- **Loads**: Power consumption (buildings, industry, etc.)
- **Links**: Connections that transfer power between buses (can model conversion efficiencies)
- **Lines**: Specific links relating to electrical transmission
- **StorageUnits**: Energy storage devices (batteries, pumped hydro)
- **Stores**: Energy reservoirs (can represent exports, imports, or long-term storage)

### 4. Carriers
Energy types or technologies (electricity, solar, wind, gas) that allow tracking of attributes like CO₂ emissions across the network.

### 5. Optimization Types
- **Dispatch optimization**: Given fixed capacities, determine optimal operation over time
- **Capacity expansion**: Determine optimal capacities to install (in addition to optimal operation)

In [1]:
# start by importing pypsa
import pypsa
import pandas as pd

In [2]:
# create empty network object using pypsa.Network()
n = pypsa.Network()

In [3]:
# create 7 buses at 120V AC using the v_nom parameter (PyPSA uses kV as the default unit for voltage). 
# See https://docs.pypsa.org/latest/user-guide/components/buses/
for i in range(7):
    n.add("Bus", f"Bus {i+1}", v_nom=0.12)

In [4]:
# add lines between the buses
# See https://docs.pypsa.org/latest/user-guide/components/lines/
for i in range(6):
    n.add("Line", f"Line {i+1}-{i+2}", bus0=f"Bus {i+1}", bus1=f"Bus {i+2}",
              x=1e-6, r=0.37, s_nom=1
    )

In [5]:
# add generator at bus 1 as a slack 
# See https://docs.pypsa.org/latest/user-guide/components/generators/
n.add("Generator", "Generator 1", bus="Bus 1", control="Slack")

In [6]:
# add loads of 300 W (pf=0.95) at each bus except the first bus
# See https://docs.pypsa.org/latest/user-guide/components/loads/
pf = 0.95
s = 300 / 1000**2  # convert W to MW
p = s * pf
q = s * (1 - pf**2)**0.5
for i in range(1, 7): # remember that python uses 0-based indexing, so bus 2 is index 1, bus 3 is index 2, etc.
    n.add("Load", f"Load {i+1}", bus=f"Bus {i+1}", p_set=p, q_set=q)

In [7]:
# we can summarise the network by printing it
n

PyPSA Network 'Unnamed Network'
-------------------------------
Components:
 - Bus: 7
 - Generator: 1
 - Line: 6
 - Load: 6
Snapshots: 1

In [8]:
# Perform the full AC power flow calculations
n.pf()

INFO:pypsa.network.power_flow:Performing non-linear load-flow on AC sub-network <pypsa.networks.SubNetwork object at 0x0000020228229BB0> for snapshots Index(['now'], dtype='object', name='snapshot')


{'n_iter': name      0
 snapshot   
 now       3,
 'error': name                 0
 snapshot              
 now       5.291410e-09,
 'converged': name         0
 snapshot      
 now       True}

In [9]:
# check the bus voltages
n.buses_t['v_mag_pu'].T

snapshot,now
name,
Bus 1,1.000000
Bus 2,0.948698
Bus 3,0.905320
Bus 4,0.870189
Bus 5,0.843582
Bus 6,0.825717
Bus 7,0.816746
